<a href="https://colab.research.google.com/github/tzs69/BT4012-kaggle-comp/blob/main/BT4012_competition_final_ensemble_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Modelling

Final Ensemble Modelling notebook where I combine the predictions of supervised models trained on the labelled data (from `BT4012_competition_baseline_modelling.ipynb`) with those obtained from the illicit vs rest modelling runs (from `BT4012_competition_unlabelled.ipynb`).

## Setup


In [4]:
%pip install -q catboost xgboost lightgbm


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive
from joblib import load, dump
from scipy.special import expit, logit
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
drive.mount('/content/drive')

ROOT_DIR = "/content/drive/MyDrive/BT4012_competition"
df_train = pd.read_csv(ROOT_DIR + "/transformed_datasets/train_topo.csv")
df_test = pd.read_csv(ROOT_DIR + "/transformed_datasets/test_topo.csv")

CACHE_DIR = ROOT_DIR + "/base_models_cache"
SUBMISSIONS_DIR = ROOT_DIR + "/submissions"

df_test = pd.read_csv(
    ROOT_DIR + "/transformed_datasets/test_topo.csv",
    usecols=["txId", "time_step", "index"],
)
edges = pd.read_csv(ROOT_DIR + "/base_datasets/txs_edgelist.csv")
sample_submission = pd.read_csv(ROOT_DIR + "/base_datasets/sample_submission.csv")

Mounted at /content/drive


## Load Saved Model Bundles and Predictions


In [5]:
cached = {}
for name in [
    "xgb", "catboost", "rf", "logreg", "lgbm", # supervised (labelled data only) models
    "xgb_ir", "lgbm_ir", "catboost_ir" # illicit v rest (incl. unlabelled data) models
]:
    path = Path(CACHE_DIR + f"/{name}.joblib")
    if not path.exists():
        continue

    bundle = load(path)
    cached[name] = {
        "oof": bundle["oof"].set_index("txId"),
        "test": bundle["test_predictions"].set_index("txId"),
        "feature_cols": bundle.get("feature_cols", []),
    }
del bundle

## Helper Functions


### Calculate incoming neighbour risk for each node

How risk is calculated for a transaction node:
1) For all its incoming neighbors (outgoing to the node), take the logits of their predicted illicit probabilies (if exists) and avg them
2) Calculate risk direction (`risk_dir = averaged incoming neighbour logits - own logit`)
3) Calculate coverage (`coverage = number of incoming neighbours with preds / total number of incoming neighbours`)
4) Compute final adjusted logit of node with incoming neighbour risk accounted for (`adjusted_logit = own_logit + risk_weight * coverage * risk_dir`)
5) Convert adjusted logits back to prediction probabilities and update all assoc. txns. Transactions with no `adjusted_logit` (meaning got no incoming neighbour or all src edges to it have no preds), retain their original pred probability.

In [6]:
incoming_counts = edges.groupby("txId2").size()

def incoming_risk(predictions, risk_weight):
    if risk_weight == 0:
        return predictions.copy()

    clipped = predictions.clip(1e-6, 1 - 1e-6) # enforce min/max range for smooth logit conversion
    own_logit = pd.Series(logit(clipped), index=predictions.index)

    neighbour_logits = pd.DataFrame({
        "txId": edges["txId2"],
        "logit": edges["txId1"].map(own_logit),
    })

    # Group incoming neighbours by destination txn and get aggregated logits sum/counts values
    grouped = neighbour_logits.groupby("txId")["logit"]
    neighbours = pd.DataFrame(index=predictions.index)
    neighbours["sum"] = grouped.sum()
    neighbours["count"] = grouped.count()

    # Transactions without available neighbour predictions (all src txns unlabelled || no src txns) get 0
    neighbours = neighbours.fillna(0)

    # Calculate risk direction
    mean_neighbour_logit = neighbours["sum"] / neighbours["count"]
    risk_dir = mean_neighbour_logit - own_logit

    # Calculate incoming neighbour coverage
    total = incoming_counts.reindex(predictions.index, fill_value=0).clip(lower=1)
    coverage = neighbours["count"] / total

    # Compute final risk-aware adjusted logit
    adjusted_logit = own_logit + (risk_weight * coverage * risk_dir)

    # Convert log odds back to pred probs betw [0, 1] and update all txn rows which have adjusted preds available
    result = predictions.copy()
    supported = neighbours["count"].gt(0)
    result.loc[supported] = expit(adjusted_logit.loc[supported])
    return result

### Combine supervised and illicit-versus-rest predictions
For each transaction:
1. Take the supervised branchs pred probs and the ir branchs pred probs and convert both to logits (with small smoothing to avoid resultant 0/1 values)
2. Multiply each branch log odds by their respective weights to produce final combined logits
    -  Formula: `combined_logits = (sup_weight * sup_logits) + (ir_weight * ir_logits)` where `sup_weight + ir_weight = 1`
3. Convert `combined_weights` back into probability values betw [0, 1] to produce final ensemble predictions

In [7]:
def combine_branches(sup, ir, ir_weight):
    if ir_weight == 0:
        return sup.copy()
    if ir_weight == 1:
        return ir.copy()

    # Convert each branchs prediction probabilities into smoothed logits
    smoothing = 1e-4
    sup_logits = np.log((sup + smoothing) / (1 - sup + smoothing))
    ir_logits = np.log((ir + smoothing) / (1 - ir + smoothing))

    # Multiply each branchs logits with assoc. branch weights and sum the results to get combined logits
    sup_weight = 1 - ir_weight
    combined_logits = (sup_weight * sup_logits) + (ir_weight * ir_logits)

    # Convert back to probabilities betw [0, 1]
    combined_probs = expit(combined_logits)

    return pd.Series(combined_probs, index=sup.index)

### Score folds helper for temporal validation

In [8]:
oof_details = cached["xgb"]["oof"][["time_step", "label", "fold"]]
oof_details = oof_details.loc[oof_details["fold"].gt(0)].copy()

def score_folds(pred):
    rows = []
    for fold, group in oof_details.groupby("fold"):
        target = group["label"].astype(int)
        values = pred.loc[group.index]
        rows.append({
            "fold": int(fold),
            "auc": roc_auc_score(target, values),
            "ap": average_precision_score(target, values),
        })
    return pd.DataFrame(rows).set_index("fold")

## Experiment Configuration

| Setting | Meaning |
|---|---|
| `ir_blend_weight = 0` | Supervised branch only |
| `ir_blend_weight = 0.5` | Equal log-odds blend of the two branches |
| `ir_blend_weight = 0.8` | 80% IR branch / 20% supervised branch in log-odds space |
| `ir_blend_weight = 1` | IR branch only |
| `risk_weight = 0` | Neighbour correction off |
| `risk_weight = 0.25` | One incoming correction with maximum neighbour weight 25% |

In [19]:
CONFIG = {
    "sup_weights": {"xgb": 0.75, "catboost": 0.25}, # all sum up to 1
    "ir_weights": {"xgb_ir": 0.6, "lgbm_ir": 0.1, "catboost_ir": 0.3}, # all sum up to 1
    "ir_blend_weight": 0.8,
    "risk_weight": 0.3,
}

assert sum([weight for weight in CONFIG["sup_weights"].values()]) == 1
assert sum([weight for weight in CONFIG["ir_weights"].values()]) == 1
assert 0 <= CONFIG["ir_blend_weight"] <= 1
assert 0 <= CONFIG["risk_weight"] <= 1

sup_weights = pd.Series(CONFIG["sup_weights"], dtype=float)
ir_weights = pd.Series(CONFIG["ir_weights"], dtype=float)
ir_blend_weight = CONFIG["ir_blend_weight"]
risk_weight = CONFIG["risk_weight"]

sup_name = "_".join(f"{name}{100 * value:g}" for name, value in sup_weights.items())
ir_name = "_".join(f"{name}{100 * value:g}" for name, value in ir_weights.items())
run_name = f"sup_{sup_name}__ir_{ir_name}__mix{100 * ir_blend_weight:g}_risk{100 * risk_weight:g}"
print(f"Experiment: {run_name}")

Experiment: sup_xgb75_catboost25__ir_xgb_ir60_lgbm_ir10_catboost_ir30__mix80_risk30


## Temporal Validation
For calculating incoming neighbours risk:
- Can evaluate as a whole; no need to split by fold and calculate as previously alr established during EDA that each timestep is a weakly connected component => no cross-temporal leakage


In [20]:
# Align each model's OOF predictions by transaction ID.
oof_preds = pd.DataFrame({
    name: data["oof"].loc[oof_details.index, "prediction"]
    for name, data in cached.items()
})

# Get weight-multiplied oof preds for each branch (for temporal validation)
sup_oof_preds = oof_preds.loc[:, sup_weights.index].dot(sup_weights)
ir_oof_preds = oof_preds.loc[:, ir_weights.index].dot(ir_weights)
blend_oof_preds = combine_branches(sup_oof_preds, ir_oof_preds, ir_blend_weight)

# Apply incoming-neighbour risk
final_ensemble_oof_preds = incoming_risk(blend_oof_preds, risk_weight).rename("prediction")

# Display
candidates = {
    "xgb": oof_preds["xgb"],
    "supervised": sup_oof_preds,
    "ir": ir_oof_preds,
    "sup_ir_blend": blend_oof_preds,
}
if risk_weight != 0:
    candidates["sup_ir_blend_risk"] = final_ensemble_oof_preds

fold_scores = {name: score_folds(pred) for name, pred in candidates.items()}
comparison = pd.DataFrame({
    name: {
        **{f"fold_{fold}_auc": row["auc"] for fold, row in scores.iterrows()},
        "mean_auc": scores["auc"].mean(),
        "mean_ap": scores["ap"].mean(),
    }
    for name, scores in fold_scores.items()
}).T

display(comparison)


,fold_1_auc,fold_2_auc,fold_3_auc,mean_auc,mean_ap
xgb,0.959041,0.994148,0.997770,0.983653,0.949798
supervised,0.949888,0.995078,0.998266,0.981077,0.949691
ir,0.857251,0.993939,0.998447,0.949879,0.934724
sup_ir_blend,0.883577,0.996436,0.999052,0.959689,0.942064
sup_ir_blend_risk,0.882838,0.996949,0.999132,0.959639,0.943001


## Generate Final Ensemble Predictions
and save to gdrive for submission


In [21]:
# Align each model's test predictions by transaction ID.
test_preds = pd.DataFrame({
    name: data["test"].loc[df_test.set_index("txId").index, "prediction"]
    for name, data in cached.items()
})

# Get weight-multiplied test preds for each branch (for final ensemble pred calculation)
sup_test_preds = test_preds.loc[:, sup_weights.index].dot(sup_weights)
ir_test_preds = test_preds.loc[:, ir_weights.index].dot(ir_weights)
blend_test_preds = combine_branches(sup_test_preds, ir_test_preds, ir_blend_weight)

# Apply incoming neighbour risk to get final ensemble test preds
final_ensemble_test_preds = incoming_risk(blend_test_preds, risk_weight).rename("prediction")

# Save to gdrive for submission
submission_ids = sample_submission["index"].map(df_test.set_index("index")["txId"])

final_ensemble_test_preds_submission = sample_submission.copy()
final_ensemble_test_preds_submission["target"] = final_ensemble_test_preds.loc[submission_ids].to_numpy()

submission_path = SUBMISSIONS_DIR + f"/{run_name}_submission.csv"
final_ensemble_test_preds_submission.to_csv(submission_path, index=False)
print(f"Submission: {submission_path}")

Submission: /content/drive/MyDrive/BT4012_competition/submissions/sup_xgb75_catboost25__ir_xgb_ir60_lgbm_ir10_catboost_ir30__mix80_risk30_submission.csv


##Leaderboard Score Tracking
| Score | Supervised Config | IR Config | Risk |
|---|---|---|---|
| 0.95301 | XGB 0.75, CatBoost 0.25; Branch weight: 0.50 | XGB 1.0; Branch weight: 0.50 | 0.0 |
| 0.95403 | XGB 0.75, CatBoost 0.25; Branch weight: 0.50 | XGB 0.75, LGBM 0.25; Branch weight: 0.50 | 0.0 |
| 0.95972 | XGB 0.75, CatBoost 0.25; Branch weight: 0.20 | XGB 0.70, LGBM 0.20, CatBoost 0.10; Branch weight: 0.80 | 0.30 |
| 0.95960 | XGB 0.65, LGBM 0.25, CatBoost 0.10; Branch weight: 0.175 | XGB 0.70, LGBM 0.20, CatBoost 0.10; Branch weight: 0.825 | 0.25 |
| 0.95968 | XGB 0.75, CatBoost 0.25; Branch weight: 0.175 | XGB 0.70, LGBM 0.20, CatBoost 0.10; Branch weight: 0.825 | 0.25 |
| 0.95904 | XGB 0.75, CatBoost 0.25; Branch weight: 0.175 | XGB 0.75, LGBM 0.25; Branch weight: 0.825 | 0.25 |
| 0.95882 | XGB 0.75, CatBoost 0.25; Branch weight: 0.25 | XGB 0.75, LGBM 0.25; Branch weight: 0.75 | 0.25 |
| 0.95849 | XGB 0.75, CatBoost 0.25; Branch weight: 0.30 | XGB 0.75, LGBM 0.25; Branch weight: 0.70 | 0.25 |
| 0.95891 | XGB 0.75, CatBoost 0.25; Branch weight: 0.0 |XGB 0.70, LGBM 0.15, CatBoost 0.15; Branch weight: 1.0 | 0.30|
| 0.95955 | XGB 0.80, Random Forest 0.20; Branch weight: 0.20 |XGB 0.70, LGBM 0.20, CatBoost 0.10; Branch weight: 0.80 | 0.30|
| 0.96019 | XGB 0.75, CatBoost 0.25; Branch weight: 0.20 |XGB 0.60, LGBM 0.10, CatBoost 0.30; Branch weight: 0.80 | 0.30|
| 0.95991 | XGB 0.75, CatBoost 0.25; Branch weight: 0.20 |XGB 0.55, LGBM 0.10, CatBoost 0.35; Branch weight: 0.80 | 0.30|
| 0.95923 | XGB 0.75, CatBoost 0.25; Branch weight: 0.20 |XGB 0.60, LGBM 0.10, CatBoost 0.30; Branch weight: 0.80 | 0.15|